# Project 5: Inventory Demand Forecasting
## Notebook 1: Dataset Collection, Data Cleaning & Preprocessing

**Course:** CSAI - Machine Learning Evaluation (Phase 1)  
**Dataset:** Walmart M5 Forecasting - Accuracy  
**Module Focus:**
- Ingestion from `data/raw/` (calendar, sell_prices, sales_train_validation).
- Schema integrity, completeness, and downcasting (~75% RAM reduction).
- Product lifecycle evaluation: Cold-start detection and pre-launch identification.
- **Fair handling of stockouts:** Marking rather than imputing stockout periods.
- Feature engineering: Cyclical calendar encodings and causal lags (without lookahead).
- Exporting structured, memory-optimized tables to `data/processed/`.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add project root to sys.path
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_calendar, load_sell_prices, load_sales_train, reduce_mem_usage
from src.data_quality import M5DataQualityAuditor
from src.data_cleaning import M5DataCleaner

print("Data Cleaning environment initialized.")


Data Cleaning environment initialized.


### 1. Raw Data Ingestion (`data/raw/`)
We load the raw CSV files and inspect schema dimensions.
- `calendar.csv`: 1,969 days with calendar attributes, SNAP dates, and events.
- `sell_prices.csv`: 6,841,121 weekly price observations per store and item.
- `sales_train_validation.csv`: 30,490 time series over 1,913 days (58,327,370 unit observations).


In [2]:
raw_dir = Path("../data/raw")

df_cal_raw = pd.read_csv(raw_dir / "calendar.csv")
df_prices_raw = pd.read_csv(raw_dir / "sell_prices.csv")
df_sales_raw = pd.read_csv(raw_dir / "sales_train_validation.csv")

print(f"Raw Calendar: {df_cal_raw.shape}")
print(f"Raw Prices:   {df_prices_raw.shape}")
print(f"Raw Sales:    {df_sales_raw.shape}")


Raw Calendar: (1969, 14)
Raw Prices:   (6841121, 4)
Raw Sales:    (30490, 1919)


### 2. Comprehensive Data Quality Audit
We verify:
1. **Completeness:** 0 null values across all 58.3 million unit sales entries.
2. **Validity:** Strictly non-negative integer sales and positive prices ($0.01 to $107.32).
3. **Continuity:** 1,969 consecutive days without date gaps.


In [3]:
auditor = M5DataQualityAuditor(raw_dir="../data/raw", reports_dir="../reports")
cal_audit = auditor.audit_calendar(load_calendar(str(raw_dir / "calendar.csv")))
price_audit = auditor.audit_sell_prices(load_sell_prices(str(raw_dir / "sell_prices.csv")))
sales_audit = auditor.audit_sales_train(load_sales_train(str(raw_dir / "sales_train_validation.csv")))

print(f"Total Observations Audited: {sales_audit['total_observations']:,}")
print(f"Sales Missing Values:       {sales_audit['data_cleanliness']['missing_values']}")
print(f"Negative Sales Detected:    {sales_audit['data_cleanliness']['negative_sales']}")
print(f"Negative Prices Detected:   {price_audit['anomalies']['negative_prices']}")


Total Observations Audited: 58,327,370
Sales Missing Values:       0
Negative Sales Detected:    0
Negative Prices Detected:   0


### 3. Product Lifecycle, Cold Starts & Stockout Marking
In retail demand modeling:
- **Pre-launch Inactivity vs. Zero Customer Demand:** Items introduced late in the 5-year timeline have leading zeros before their first launch. Misclassifying these as 'zero demand' biases models.
- **Cold-Start Items:** Items active for less than 1 year (365 days) require global representation sharing.
- **Fair Stockout Handling:** Rather than imputing missing sales, we construct an explicit `is_unlisted_stockout` indicator for periods where an active item disappears from store price listings.


In [4]:
cleaner = M5DataCleaner(raw_dir="../data/raw", processed_dir="../data/processed")
sku_meta = cleaner.analyze_lifecycle_and_stockouts(df_sales_raw)

print(f"Total SKUs Analyzed:        {len(sku_meta):,}")
print(f"Cold-Start SKUs (<365 days): {sku_meta['is_cold_start'].sum():,} ({sku_meta['is_cold_start'].mean()*100:.2f}%)")
display(sku_meta.head(10))


[CLEANING] Evaluating product lifecycle, cold-starts, and stockout markers...


Total SKUs Analyzed:        30,490
Cold-Start SKUs (<365 days): 428 (1.40%)


,id,item_id,dept_id,cat_id,store_id,state_id,first_sale_day,first_sale_day_num,active_days,is_cold_start
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_902,902,1012,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_144,144,1770,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1106,1106,808,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_37,37,1877,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_113,113,1801,0
5,HOBBIES_1_006_CA_1_validation,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,d_430,430,1484,0
6,HOBBIES_1_007_CA_1_validation,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA,d_531,531,1383,0
7,HOBBIES_1_008_CA_1_validation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,d_1,1,1913,0
8,HOBBIES_1_009_CA_1_validation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,d_1,1,1913,0
9,HOBBIES_1_010_CA_1_validation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,d_3,3,1911,0


### 4. End-to-End Pipeline Execution & CSV Export (`data/processed/`)
We run the full cleaning pipeline and export memory-optimized CSV tables into `data/processed/`:
1. `cleaned_calendar.csv`: Cyclical day/month trigonometric features, SNAP flags, Christmas marker.
2. `cleaned_prices.csv`: Regular price baselines, discount depth percentages, promo markers.
3. `sku_lifecycle_metadata.csv`: Cold-start flags and active shelf duration.
4. `cleaned_sales_matrix.csv`: Downcasted sales matrix.
5. `m5_starter_modeling_panel.csv`: Complete panel for Store `CA_1` `FOODS_1` (50 SKUs) with causal lag and rolling features without lookahead.


In [5]:
summary = cleaner.run_cleaning_pipeline()

import os
processed_files = os.listdir("../data/processed")
print("\nGenerated Clean Files in data/processed/:")
for f in sorted(processed_files):
    sz = (Path("../data/processed") / f).stat().st_size / (1024*1024)
    print(f"  • {f:<35} ({sz:.2f} MB)")


  M5 DATA CLEANING & PREPROCESSING PIPELINE


[CLEANING] Processing calendar data...
[SAVED] Cleaned Calendar -> cleaned_calendar.csv ((1969, 22))
[CLEANING] Processing price dynamics and discount features...


[SAVED] Cleaned Prices -> cleaned_prices.csv & sample ((6841121, 7))
[CLEANING] Evaluating product lifecycle, cold-starts, and stockout markers...


[SAVED] SKU Lifecycle Metadata -> sku_lifecycle_metadata.csv ((30490, 10))
[CLEANING] Optimizing sales matrix for processed storage...


[SAVED] Cleaned Sales Matrix -> cleaned_sales_matrix.csv & sample ((30490, 1919))
[CLEANING] Generating starter modeling panel for CA_1 / FOODS_1 (top 50 SKUs)...


[CLEANING] Engineering causal lag and rolling features (no forward-looking leakage)...


[SAVED] Starter Modeling Panel -> m5_starter_modeling_panel.csv ((95650, 35))

[DONE] Data Cleaning pipeline completed in 29.50 seconds.
Cleaned datasets safely stored in: /Users/dakarapuhemamdharnath/Desktop/AMLPROJECT/data/processed



Generated Clean Files in data/processed/:
  • .gitkeep                            (0.00 MB)
  • README.md                           (0.00 MB)
  • cleaned_calendar.csv                (0.20 MB)
  • cleaned_prices.csv                  (307.29 MB)
  • cleaned_prices_sample.csv           (0.22 MB)
  • cleaned_sales_matrix.csv            (114.45 MB)
  • cleaned_sales_sample.csv            (1.88 MB)
  • m5_starter_modeling_panel.csv       (18.90 MB)
  • sku_lifecycle_metadata.csv          (2.45 MB)
